# Logistic Regression with PyTorch

In this notebook you will implement a **logistic regression** classifier in PyTorch and apply it to a non-linear classification problem.

The building blocks of PyTorch are Tensors and Operations. With these we can form dynamic computational graphs that represent models. Here, we define a logistic regression model using these simple building blocks and train it on a 2D binary classification problem.

### Important note
Logistic regression is a **linear classifier** — it can only separate classes with a straight line (or hyperplane in higher dimensions). The dataset we'll use has a non-linear decision boundary, so **we already know upfront that logistic regression will struggle**. That's intentional: observing where and why a model fails is just as instructive as watching it succeed.

In this notebook you should:
* **First** run the code as is, and observe the model's behavior.
* **Then** complete the assignments at the bottom of the notebook.
* **Lastly** reflect on the limitations you've observed and think about how they might be overcome.

> We assume that you are already familiar with backpropagation (if not, see [Andrej Karpathy](http://cs.stanford.edu/people/karpathy/) or [Michael Nielsen](http://neuralnetworksanddeeplearning.com/chap2.html)).

# Dependencies and supporting functions
Load dependencies and supporting functions by running the code block below.

In [ ]:
%matplotlib inline
import matplotlib
import numpy as np
import matplotlib.pyplot as plt
import sklearn.datasets

# Do not worry about the code below for now, it is used for plotting later
def plot_decision_boundary(pred_func, X, y):
    #from https://github.com/dennybritz/nn-from-scratch/blob/master/nn-from-scratch.ipynb
    # Set min and max values and give it some padding
    x_min, x_max = X[:, 0].min() - .5, X[:, 0].max() + .5
    y_min, y_max = X[:, 1].min() - .5, X[:, 1].max() + .5
    
    h = 0.01
    # Generate a grid of points with distance h between them
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    
    yy = yy.astype('float32')
    xx = xx.astype('float32')
    # Predict the function value for the whole grid
    Z = pred_func(np.c_[xx.ravel(), yy.ravel()])[:,0]
    Z = Z.reshape(xx.shape)
    # Plot the contour and training examples
    plt.figure()
    plt.contourf(xx, yy, Z, cmap=plt.cm.RdBu)
    plt.scatter(X[:, 0], X[:, 1], c=-y, cmap=plt.cm.Spectral)

def onehot(t, num_classes):
    out = np.zeros((t.shape[0], num_classes))
    for row, col in enumerate(t):
        out[row, col] = 1
    return out

# The Problem: Half-Moon Dataset

We'll work on the **half-moon dataset** — two interleaving crescent shapes that cannot be separated by a straight line. This makes it a deliberately hard problem for logistic regression.

As you'll see, logistic regression will only produce a linear decision boundary, and will therefore be unable to correctly classify all samples. This is a fundamental limitation of the model, not a bug in the implementation.

In [ ]:
# Generate a dataset and plot it
np.random.seed(0)
num_samples = 300

X, y = sklearn.datasets.make_moons(num_samples, noise=0.20)

# define train, validation, and test sets
X_tr = X[:100].astype('float32')
X_val = X[100:200].astype('float32')
X_te = X[200:].astype('float32')

# and labels
y_tr = y[:100].astype('int32')
y_val = y[100:200].astype('int32')
y_te = y[200:].astype('int32')

plt.scatter(X_tr[:,0], X_tr[:,1], s=40, c=y_tr, cmap=plt.cm.Spectral)
plt.title("Half-Moon Training Data")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.show()

print("Data shape:", X.shape, y.shape)

num_features = X_tr.shape[-1]
num_output = 2

print(f"Number of input features: {num_features}")
print(f"Number of output classes: {num_output}")

# Logistic Regression

Logistic regression is the simplest classification model. It applies a single **linear transformation** followed by a **softmax** activation:

$$\hat{y} = \text{softmax}(W^{\top} x + b)$$

where:
- $x$ is the input vector of shape `[batch_size, num_features]`
- $W$ is a weight matrix of shape `[num_output, num_features]`
- $b$ is a bias vector of shape `[num_output]`
- $\hat{y}$ is the predicted class probability vector of shape `[batch_size, num_output]`

The decision boundary of this model is always a **straight line** (in 2D) or a hyperplane (in higher dimensions). No matter how long you train it, it will never learn a curved boundary.

## PyTorch 101

We'll use basic PyTorch building blocks so you can see exactly what's happening under the hood. This also builds intuition for when you want to define custom operations later.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F

[`Parameters`](https://pytorch.org/docs/stable/generated/torch.nn.parameter.Parameter.html) have a special property: when assigned as attributes of a [`Module`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html), they are automatically registered and appear in `model.parameters()`. This is what makes automatic gradient tracking and optimizer updates work seamlessly.

In [ ]:
class LogisticRegression(nn.Module):

    def __init__(self):
        super(LogisticRegression, self).__init__()
        # A single linear layer: maps num_features inputs to num_output outputs
        # W shape: [num_output, num_features] = [2, 2]
        # b shape: [num_output]               = [2]
        self.W = nn.Parameter(torch.randn(num_output, num_features))
        self.b = nn.Parameter(torch.randn(num_output))
        
    def forward(self, x):
        # Linear transformation: z = Wx + b
        x = F.linear(x, self.W, self.b)
        # Softmax converts raw scores to class probabilities
        return F.softmax(x, dim=1)

net = LogisticRegression()
print(net)

### Inspecting Parameters

In [ ]:
# List all named parameters in the network
print("NAMED PARAMETERS")
print(list(net.named_parameters()))
print()

print("WEIGHTS")
print(net.W)
print("Shape:", net.W.size())
print('\nBIAS')
print(net.b)
print("Shape:", net.b.size())

# Exploring Parameters and Gradients

Before training, let's understand the structure of a `Parameter` — specifically the `.data` tensor and the `.grad` tensor that gets populated during backpropagation.

In [ ]:
param = net.W
print("## The weight tensor")
print(param.data)
print("\n## Its gradient (None before backward pass)")
print(param.grad)
print("\n## Is it a leaf in the computation graph?")
print(param.is_leaf)

## Excluding subgraphs from backward propagation

To exclude part of the computation graph from gradient tracking, set `requires_grad=False` on the relevant tensors. An operation's output requires a gradient only if at least one of its inputs does.

# Test the Network (Forward Pass)

Let's run the network on a small batch of random inputs to verify it works before training.

In [ ]:
X_test = torch.randn(5, num_features)

print('Input (5 random samples):')
print(X_test)

print('\nOutput (class probabilities):')
print(net(X_test))
print('\nNote: each row sums to 1.0 (softmax output)')
print('Row sums:', net(X_test).sum(dim=1).data)

### Gradients before and after a backward pass

In [ ]:
# Gradients are None before any backward pass
print("Before backward:")
for p in net.parameters():
    print(" data:", p.data)
    print(" grad:", p.grad)
    print()

In [ ]:
X_test2 = torch.randn(7, num_features)
out = net(X_test2)
# Pass dummy gradients to trigger backprop
out.backward(torch.randn(7, num_output))

print("After backward:")
for p in net.parameters():
    print(" data:", p.data)
    print(" grad:", p.grad)
    print()

In [ ]:
# Zero out accumulated gradients before each training step
net.zero_grad()
print("After zero_grad():")
for p in net.parameters():
    print(" grad:", p.grad)

# Loss Function

We use **cross-entropy loss**, which measures how far the predicted probability distribution is from the true one-hot labels:

$$\mathcal{L} = -\frac{1}{N} \sum_{i=1}^{N} \sum_{c=1}^{C} t_{ic} \log(\hat{y}_{ic})$$

In [ ]:
def cross_entropy(ys, ts):
    # Cross entropy per sample
    cross_entropy = -torch.sum(ts * torch.log(ys), dim=1, keepdim=False)
    # Average over samples
    return torch.mean(cross_entropy)

To train, we update the parameters in the direction of the negative gradient of the loss. We use [`torch.optim`](http://pytorch.org/docs/master/optim.html) to handle this with a chosen update rule.

In [ ]:
import torch.optim as optim

optimizer = optim.SGD(net.parameters(), lr=0.01)

In [ ]:
def accuracy(ys, ts):
    # One-hot encoded vector of correct predictions
    correct_prediction = torch.eq(torch.max(ys, 1)[1], torch.max(ts, 1)[1])
    # Average to get accuracy
    return torch.mean(correct_prediction.float())

# Training the Logistic Regression Model

We now train the model for 1000 epochs. Watch the loss and accuracy curves — notice that even after full convergence, the model cannot perfectly separate the two moons. This is the fundamental limitation we set out to observe.

In [ ]:
# Re-initialize the model and optimizer cleanly
net = LogisticRegression()
optimizer = optim.SGD(net.parameters(), lr=0.01)

num_epochs = 1000
train_losses, val_losses, train_accs, val_accs = [], [], [], []

def pred(X):
    """Run forward pass and return numpy array of predictions."""
    X = torch.from_numpy(X)
    y = net(X)
    return y.data.numpy()

# Visualize decision boundary before training
plot_decision_boundary(lambda x: pred(x), X_te, y_te)
plt.title("Untrained Logistic Regression")
plt.show()

# Training loop
for e in range(num_epochs):
    tr_input = torch.from_numpy(X_tr)
    tr_targets = torch.from_numpy(onehot(y_tr, num_output)).float()
    
    optimizer.zero_grad()
    tr_output = net(tr_input)
    tr_loss = cross_entropy(tr_output, tr_targets)
    tr_loss.backward()
    optimizer.step()
    train_acc = accuracy(tr_output, tr_targets)
    
    train_losses.append(tr_loss.data.numpy())
    train_accs.append(train_acc)
    
    val_input = torch.from_numpy(X_val)
    val_targets = torch.from_numpy(onehot(y_val, num_output)).float()
    val_output = net(val_input)
    val_loss = cross_entropy(val_output, val_targets)
    val_acc = accuracy(val_output, val_targets)
    
    val_losses.append(val_loss.data.numpy())
    val_accs.append(val_acc.data.numpy())
    
    if e % 100 == 0:
        print("Epoch %i,  Train Loss: %0.3f\tVal Loss: %0.3f\tVal Acc: %0.3f" % (
            e, train_losses[-1], val_losses[-1], val_accs[-1]))

# Evaluate on test set
te_input = torch.from_numpy(X_te)
te_targets = torch.from_numpy(onehot(y_te, num_output)).float()
te_output = net(te_input)
te_loss = cross_entropy(te_output, te_targets)
te_acc = accuracy(te_output, te_targets)
print("\nTest Loss: %0.3f\tTest Accuracy: %0.3f" % (te_loss.data.numpy(), te_acc.data.numpy()))

# Visualize learned decision boundary
plot_decision_boundary(lambda x: pred(x), X_te, y_te)
plt.title("Trained Logistic Regression (Linear Boundary)")
plt.show()

# Loss curves
epoch = np.arange(len(train_losses))
plt.figure()
plt.plot(epoch, train_losses, 'r', label='Train Loss')
plt.plot(epoch, val_losses, 'b', label='Val Loss')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.show()

# Accuracy curves
plt.figure()
plt.plot(epoch, train_accs, 'r', label='Train Acc')
plt.plot(epoch, val_accs, 'b', label='Val Acc')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.show()

# Assignments

Work through the following assignments in order. Each one builds on the last.

---

### Assignment 1 — Inspect the learned boundary
Look at the decision boundary plot after training. Can you describe the shape of the boundary? Why is it shaped that way? What does this tell you about what logistic regression can and cannot learn?

*Hint: think about the equation $\hat{y} = \text{softmax}(Wx + b)$ and what kind of surfaces the equation $Wx + b = 0$ can represent.*

---

### Assignment 2 — Change the learning rate
Try different values for the learning rate (e.g. `0.001`, `0.1`, `1.0`) and re-run training. How does it affect:
- The speed of convergence?
- The final loss and accuracy?
- The stability of training?

---

### Assignment 3 — Change the optimizer
Replace `optim.SGD` with [`optim.Adam`](https://pytorch.org/docs/master/optim.html#torch.optim.Adam) and re-run training with the same number of epochs. Does Adam converge faster? Does it reach a better final accuracy? Can logistic regression with Adam learn a curved boundary?

---

### Assignment 4 — Try a different dataset
Replace `make_moons` with [`make_circles`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_circles.html) from scikit-learn:
```python
X, y = sklearn.datasets.make_circles(num_samples, noise=0.10)
```
Train logistic regression on this new dataset. Can it classify circular data? Why or why not?

---

### Assignment 5 — Feature engineering (bonus)
One classical workaround for logistic regression's linear limitation is **feature engineering**: manually adding non-linear features to the input before training.

Try augmenting the input features by adding polynomial terms. For example:
```python
# Original: [x1, x2]
# Augmented: [x1, x2, x1^2, x2^2, x1*x2]
X_aug = np.hstack([X, X**2, (X[:, 0:1] * X[:, 1:2])])
```
Update `num_features` accordingly and retrain. Does the decision boundary look more curved now? What does this tell you about the relationship between feature engineering and model capacity?

In [ ]:
# Your code here
